# Module 3: Serving LLMs Efficiently with vLLM

In Module 2 you opened up a single request and measured where its time and memory go. But one request barely touches the card: a single decode step uses a sliver of the GPU, so the hardware you are paying for sits mostly idle. This module fixes that. You run the same work two ways, first one request at a time, then many at once, and watch [vLLM](https://docs.vllm.ai) pack the concurrent requests into a single running batch. That packing is **continuous batching**, and it is the single biggest reason a dedicated GPU on an [Akamai Cloud GPU](https://www.linode.com/products/gpu/) pays off.

## Learning objectives
- Explain why a single request leaves the GPU mostly idle
- Run N requests sequentially and read the aggregate throughput floor
- Fire the same N requests concurrently and measure the speedup
- Watch `vllm:num_requests_running` climb as vLLM forms one running batch
- Plot the batch growing and shrinking over the run
- Name the cost of batching, a small per-request slowdown, and why it is worth it

## Prerequisites
- Finished Modules 0, 1, and 2, with a working endpoint and resolved settings
- A self-hosted vLLM endpoint in `VLLM_HOST`, reachable from this notebook
- Comfortable reading the vLLM metrics from Module 2
- About 15 minutes

References: [Anatomy of vLLM](https://vllm.ai/blog/2025-09-05-anatomy-of-vllm) &middot; [vLLM optimization guide](https://docs.vllm.ai/en/stable/configuration/optimization/) &middot; [Continuous batching for LLM inference](https://www.anyscale.com/blog/continuous-batching-llm-inference) &middot; [vLLM metrics reference](https://docs.vllm.ai/en/stable/usage/metrics.html)

## Continuous batching design basics

A naive server is strictly sequential: it handles one request, finishes it, then starts the next. Each decode step uses a fraction of the GPU, so between requests the card is idle and aggregate throughput stays near one request's worth no matter how many you send.

vLLM runs a **continuous (in-flight) batch** instead. On every scheduler step it advances all running requests by one decode token. When a request finishes, the scheduler slots a waiting one into the freed space, even mid-flight, and new prompts get prefilled and merged into the same batch. The batch is not fixed at submit time. It grows and shrinks every step.

The result: as you add concurrent requests, total throughput (tokens per second across all of them) rises sharply, while each individual request slows only a little, until you run out of KV cache. The metric that shows the batch forming is `vllm:num_requests_running`.

![Sequential serving leaves the GPU idle between requests while continuous batching keeps one running batch full so the GPU stays busy and throughput climbs](images/03_serving_with_vllm_architecture.png)

## 1. Setup

This module needs the OpenAI client to call the server, `requests` to scrape `/metrics`, and `matplotlib` to plot the batch. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q "openai>=1.40" requests matplotlib

## 2. Configure endpoint and model

`get_settings()` reads your connection details from the environment, and `build_client()` returns an OpenAI client pointed at your vLLM. The `common.metrics` helper scrapes the vLLM `/metrics` endpoint and gives you a recorder you can sample over time. Same setup as the earlier modules, plus the metrics helper you will use to watch the batch.

In [ ]:
# Setup: imports, settings, client, and a metrics recorder.
import os, sys, time, threading
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, os.path.abspath(".."))

from common.config import get_settings, build_client
from common import metrics

settings = get_settings()
client = build_client(settings)
print("endpoint:", settings.vllm_host, "| model:", settings.model_name)

**What you should see:** your endpoint and model printed. Same connection as the earlier modules.

## 3. Define one unit of work

Define a helper that sends a single request and returns how many tokens came back and how long it took. You will call it sequentially first, then concurrently, with no other change. Same prompt every time keeps the comparison honest: the only variable is how many run at once.

In [ ]:
# One request. Returns (completion_tokens, seconds). Same prompt every time.
PROMPT = "Write a short paragraph explaining continuous batching in LLM inference."

def one_request():
    start = time.time()
    resp = client.chat.completions.create(
        model=settings.model_name,
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=200,
        temperature=0.0,
    )
    return resp.usage.completion_tokens, time.time() - start

**What you should see:** nothing yet. This defines the unit of work used by both runs below.

## 4. The slow way: requests one at a time

First, the failure to beat. Run several requests sequentially. The GPU handles one at a time, so total throughput is roughly one request's worth no matter how many you send. This is your floor, and it is the floor a naive server is stuck on.

In [ ]:
# Requires a live vLLM endpoint.
# Run N requests sequentially and measure aggregate throughput.
N = 8

start = time.time()
seq_results = [one_request() for _ in range(N)]
seq_wall = time.time() - start

seq_tokens = sum(t for t, _ in seq_results)
print(f"sequential: {N} requests in {seq_wall:.1f}s")
print(f"aggregate throughput: {seq_tokens/seq_wall:.0f} tokens/s")

**What you should see:** a wall-clock time roughly equal to N times one request, and a throughput near a single request's rate. The GPU was idle most of the time because only one request occupied it at once. That idle time is the money you are leaving on the table.

## 5. The fix: fire them at once and watch the batch form

Now send the same N requests concurrently with a thread pool, nothing else changed. While they run, a background recorder samples `/metrics` every 0.25s so you can watch `num_requests_running` climb as vLLM merges the requests into one batch. The aggregate throughput should jump well above the sequential floor.

In [ ]:
# Requires a live vLLM endpoint.
# Sample metrics in the background while N requests run concurrently.
recorder = metrics.SeriesRecorder(settings.metrics_url)
stop = threading.Event()

def sample_loop():
    while not stop.is_set():
        try:
            recorder.record()
        except Exception:
            pass
        time.sleep(0.25)

sampler = threading.Thread(target=sample_loop, daemon=True)
sampler.start()

start = time.time()
with ThreadPoolExecutor(max_workers=N) as pool:
    conc_results = list(pool.map(lambda _: one_request(), range(N)))
conc_wall = time.time() - start

stop.set()
sampler.join(timeout=2)

conc_tokens = sum(t for t, _ in conc_results)
print(f"concurrent: {N} requests in {conc_wall:.1f}s")
print(f"aggregate throughput: {conc_tokens/conc_wall:.0f} tokens/s")
print(f"speedup vs sequential: {seq_wall/conc_wall:.1f}x")
peak_running = max(recorder.series("vllm:num_requests_running"), default=0)
print(f"peak num_requests_running: {peak_running:.0f}")

**What you should see:** a much shorter wall-clock time, an aggregate throughput several times the sequential number (on a single RTX 4000 Ada with Qwen3-4B this lands near 7x), and a peak `num_requests_running` greater than 1, often close to N for a small N. That peak is the batch. Each request was a little slower than solo, but together they did far more work per second.

## 6. Plot the batch forming

The recorder captured `num_requests_running` over the run. Plot it to see the batch grow as requests arrive and shrink as they finish. This curve is continuous batching in one picture.

In [ ]:
# Plot the running batch size over the concurrent run.
ax = recorder.plot("vllm:num_requests_running", title="Continuous batch size during concurrent load")
ax.set_ylabel("requests in running batch")

**What you should see:** a curve that rises as the requests land, holds while they decode together, then falls as they complete. With only 8 requests it is a short hill. In the next module you hold the batch full and find where it breaks.

## Things to know

- **The batch is dynamic.** vLLM does not wait for a fixed batch to fill before running. It admits and evicts requests every scheduler step, which is why it is called continuous, or in-flight, batching. A request can join a batch that is already decoding.
- **Concurrency is not parallel hardware.** You did not add a second GPU. The same card now does more per step because many requests share one forward pass. Throughput went up, GPU count did not.
- **The ceiling is KV cache, not threads.** Adding more concurrent requests helps only until the KV cache fills. The gauge `kv_cache_usage_perc` tells you how close you are. vLLM's V1 engine renamed it from `gpu_cache_usage_perc`, and the metrics helper returns it under both names so your reads keep working.
- **Per-request latency rises a little.** Sharing the GPU means each request waits its turn in the batch. That is the trade: a small individual slowdown for a large throughput gain. Module 8 puts a latency SLO on it.

> NOTE: With only 8 requests the batch forms and drains quickly, so the curve is a short hill. Raise `N` to hold the batch fuller for longer, but watch `num_requests_waiting` start to climb once the running batch is saturated.

## Try it yourself

**Raise the concurrency.** Increase `N` and rerun the concurrent cell. Watch the peak `num_requests_running` and the speedup grow, then flatten once the batch saturates. **Stretch:** also print `peak num_requests_waiting` to see the queue start to form.

**Lengthen the answers.** Raise `max_tokens` in `one_request` so each request decodes longer. Longer requests stay in the batch longer, so the running batch holds higher. See how that changes the throughput gain.

**Compare floor to ceiling.** Re-run the sequential cell and the concurrent cell back to back and read the speedup line. That ratio is the throughput continuous batching buys you on your hardware.

In [ ]:
# Change N, then run the cell to re-measure the concurrent batch.
N = 16   # try 16, 32, 64 and watch the peak running batch and the speedup

recorder = metrics.SeriesRecorder(settings.metrics_url)
stop = threading.Event()

def sample_loop():
    while not stop.is_set():
        try:
            recorder.record()
        except Exception:
            pass
        time.sleep(0.25)

sampler = threading.Thread(target=sample_loop, daemon=True)
sampler.start()

start = time.time()
with ThreadPoolExecutor(max_workers=N) as pool:
    results = list(pool.map(lambda _: one_request(), range(N)))
wall = time.time() - start

stop.set()
sampler.join(timeout=2)

tokens = sum(t for t, _ in results)
print(f"concurrent: {N} requests in {wall:.1f}s -> {tokens/wall:.0f} tokens/s")
print(f"peak num_requests_running: {max(recorder.series('vllm:num_requests_running'), default=0):.0f}")
print(f"peak num_requests_waiting: {max(recorder.series('vllm:num_requests_waiting'), default=0):.0f}")

## Summary

- A single request leaves the GPU mostly idle, so sequential serving wastes the card you are paying for.
- Running the same work concurrently lets vLLM merge requests into one continuous batch, and aggregate throughput jumps several times over the sequential floor.
- `vllm:num_requests_running` shows the batch forming, and the plotted curve is continuous batching in one picture.
- The cost is a small per-request slowdown, and the ceiling is the KV cache, not the number of threads. That is the throughput win a dedicated GPU is built to capture.

## Next

**Module 4: Saturate Your GPU.** You were gentle here. Next you stop being gentle and drive rising concurrency until the batch fills, the queue grows, the KV cache nears full, and preemption starts, so you can name the real bottleneck.